# **Instalasi Library**

In [56]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# **Import Library**

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)


# **Data Loading**

Memuat dataset dari GitHub

In [3]:
app_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/raw/application_record.csv")
cred_df = pd.read_csv("https://raw.githubusercontent.com/arawsardni/5---Gaung-Taqwa-Indraswara---Sandra-Triana-Nursyafri/refs/heads/main/Dataset/raw/credit_record.csv")

# Create Target Feature

Membuat target fitur, yaitu is_bad sesuai dengan IRFS 9 (standar perbankan dunia) basel iii, 90 hari adalah kategori tunjakan serius

In [4]:
is_bad = (
    cred_df['STATUS']
    .isin(['1', '2', '3','4','5'])
    .groupby(cred_df['ID'])
    .max()  # Jika ada minimal 1 True, return 1
    .astype(int)
)

membuat cred_agg untuk menambahkan fitur months balance min dan max untuk mempermudah analisis

In [7]:
# 3. Hitung fitur temporal
credit_agg = cred_df.groupby('ID').agg(
    MONTHS_BALANCE_MIN=('MONTHS_BALANCE', 'min'),
    MONTHS_BALANCE_MAX=('MONTHS_BALANCE', 'max'),
).reset_index()

# Gabungkan target dan fitur temporal
credit_agg['IS_BAD'] = is_bad.values

In [8]:
credit_agg

,ID,MONTHS_BALANCE_MIN,MONTHS_BALANCE_MAX,IS_BAD
0,5001711,-3,0,0
1,5001712,-18,0,0
2,5001713,-21,0,0
3,5001714,-14,0,0
4,5001715,-59,0,0
...,...,...,...,...
45980,5150482,-28,-11,0
45981,5150483,-17,0,0
45982,5150484,-12,0,0
45983,5150485,-1,0,0


# Create New Dataset

In [9]:
# 4. Optimasi merge
merged_df = pd.merge(
    app_df,
    credit_agg,
    on='ID',
    how='inner'
)

In [12]:
# 4. Validasi Hasil Merge
print(f"Jumlah Data Awal (Application): {len(app_df)}")
print(f"Jumlah Data Setelah Merge: {len(merged_df)}")
print("\nDistribusi Target:")
print(merged_df['IS_BAD'].value_counts(normalize=True))

Jumlah Data Awal (Application): 438557
Jumlah Data Setelah Merge: 36457

Distribusi Target:
IS_BAD
0    0.8823
1    0.1177
Name: proportion, dtype: float64


In [13]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   36457 non-null  int64  
 1   CODE_GENDER          36457 non-null  object 
 2   FLAG_OWN_CAR         36457 non-null  object 
 3   FLAG_OWN_REALTY      36457 non-null  object 
 4   CNT_CHILDREN         36457 non-null  int64  
 5   AMT_INCOME_TOTAL     36457 non-null  float64
 6   NAME_INCOME_TYPE     36457 non-null  object 
 7   NAME_EDUCATION_TYPE  36457 non-null  object 
 8   NAME_FAMILY_STATUS   36457 non-null  object 
 9   NAME_HOUSING_TYPE    36457 non-null  object 
 10  DAYS_BIRTH           36457 non-null  int64  
 11  DAYS_EMPLOYED        36457 non-null  int64  
 12  FLAG_MOBIL           36457 non-null  int64  
 13  FLAG_WORK_PHONE      36457 non-null  int64  
 14  FLAG_PHONE           36457 non-null  int64  
 15  FLAG_EMAIL           36457 non-null 

In [14]:
merged_df.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTHS_BALANCE_MIN,MONTHS_BALANCE_MAX,IS_BAD
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,-15,0,1
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,-14,0,1
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,-29,0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,-4,0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,-26,-22,0


In [15]:
for column in merged_df.select_dtypes(include='object'):
    unique_values = merged_df[column].unique()
    print(f"Unique values in {column}: {unique_values}")

Unique values in CODE_GENDER: ['M' 'F']
Unique values in FLAG_OWN_CAR: ['Y' 'N']
Unique values in FLAG_OWN_REALTY: ['Y' 'N']
Unique values in NAME_INCOME_TYPE: ['Working' 'Commercial associate' 'Pensioner' 'State servant' 'Student']
Unique values in NAME_EDUCATION_TYPE: ['Higher education' 'Secondary / secondary special' 'Incomplete higher'
 'Lower secondary' 'Academic degree']
Unique values in NAME_FAMILY_STATUS: ['Civil marriage' 'Married' 'Single / not married' 'Separated' 'Widow']
Unique values in NAME_HOUSING_TYPE: ['Rented apartment' 'House / apartment' 'Municipal apartment'
 'With parents' 'Co-op apartment' 'Office apartment']
Unique values in OCCUPATION_TYPE: [nan 'Security staff' 'Sales staff' 'Accountants' 'Laborers' 'Managers'
 'Drivers' 'Core staff' 'High skill tech staff' 'Cleaning staff'
 'Private service staff' 'Cooking staff' 'Low-skill Laborers'
 'Medicine staff' 'Secretaries' 'Waiters/barmen staff' 'HR staff'
 'Realty agents' 'IT staff']


# Handle Missing Value

melakukan penggantian missing value di occupation type menjadi object string 'Unknown' untuk keperluan EDA

In [17]:
# Cek persentase missing values
missing_report = merged_df.isnull().mean() * 100
print("Missing Values Report:")
print(missing_report[missing_report > 0])

# Handle OCCUPATION_TYPE (kolom dengan missing tertinggi)
merged_df['OCCUPATION_TYPE'] = merged_df['OCCUPATION_TYPE'].fillna('Unknown')

print("Missing Values Report After Imputation:")
print(missing_report[missing_report > 0])

Missing Values Report:
Series([], dtype: float64)
Missing Values Report After Imputation:
Series([], dtype: float64)


# Feature Engineering

Memperoleh informasi berapa lama suatu nasabah telah berlangganan

In [68]:
# 5. Hitung fitur turunan
merged_df['CREDIT_HISTORY_LENGTH'] = (
    merged_df['MONTHS_BALANCE_MIN'] - merged_df['MONTHS_BALANCE_MAX']
)

Menambahkan fitur AGE dan YEARS_EMPLOYED yang lebih berguna saat visualisasi nanti

In [69]:
# Usia dalam tahun (DAYS_BIRTH bernilai negatif)
merged_df['AGE'] = (-merged_df['DAYS_BIRTH'] / 365).round(1)

# Lama bekerja (DAYS_EMPLOYED)
merged_df['YEARS_EMPLOYED'] = (
    merged_df['DAYS_EMPLOYED']
    .apply(lambda x: -x/365 if x < 0 else 0)
    .round(1)
)

Melakukan binning dari fitur AMT_INCOME_TOTAL menjadi INCOME_GROUP dan AGE menjadi AGE_GROUP yang diharapkan memudahkan melihat pola dlam data

In [20]:
print(merged_df['AMT_INCOME_TOTAL'].describe())
print(merged_df['AMT_INCOME_TOTAL'].isnull().sum())
min_income = merged_df['AMT_INCOME_TOTAL'].min()
max_income = merged_df['AMT_INCOME_TOTAL'].max()
print("Nilai Minimum:", min_income, "Nilai Maksimum:", max_income)


count    3.645700e+04
mean     1.866857e+05
std      1.017892e+05
min      2.700000e+04
25%      1.215000e+05
50%      1.575000e+05
75%      2.250000e+05
max      1.575000e+06
Name: AMT_INCOME_TOTAL, dtype: float64
0
Nilai Minimum: 27000.0 Nilai Maksimum: 1575000.0


In [23]:
bins = pd.cut(merged_df['AMT_INCOME_TOTAL'], bins=4, labels=['Low', 'Medium', 'High', 'Very High'])
print(bins.value_counts())


AMT_INCOME_TOTAL
Low          35437
Medium         941
High            65
Very High       14
Name: count, dtype: int64


In [70]:
# Binning pendapatan
merged_df['INCOME_GROUP'] = pd.cut(
    merged_df['AMT_INCOME_TOTAL'],
    bins=[0, 30000*7.28, 60000*7.28, 100000*7.28, np.inf],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Binning usia
merged_df['AGE_GROUP'] = pd.cut(
    merged_df['AGE'],
    bins=[18, 30, 40, 50, 60, 100],
    labels=['18-29', '30-39', '40-49', '50-59', '60+']
)

mengubah flag yang sebelumnya object menjadi binary

In [71]:
# Binary encoding untuk flag
binary_mapping = {'Y': 1, 'N': 0}
merged_df['FLAG_OWN_CAR'] = merged_df['FLAG_OWN_CAR'].map(binary_mapping)
merged_df['FLAG_OWN_REALTY'] = merged_df['FLAG_OWN_REALTY'].map(binary_mapping)



membuat label encoder untuk education level

In [72]:
# Label encoding untuk pendidikan
education_order = [
    'Lower secondary', 
    'Secondary / secondary special', 
    'Incomplete higher', 
    'Higher education', 
    'Academic degree'
]
merged_df['EDUCATION_RANK'] = merged_df['NAME_EDUCATION_TYPE'].map(
    {k:v for v,k in enumerate(education_order)}
)

membersihkan outlier

In [73]:
# Deteksi outlier AMT_INCOME_TOTAL
Q1 = merged_df['AMT_INCOME_TOTAL'].quantile(0.25)
Q3 = merged_df['AMT_INCOME_TOTAL'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

# Cap outliers
merged_df['AMT_INCOME_TOTAL_CAPPED'] = merged_df['AMT_INCOME_TOTAL'].clip(
    lower=lower_bound, 
    upper=upper_bound
)

# Atau hapus outliers
# merged_df = merged_df.query('AMT_INCOME_TOTAL >= @lower_bound & AMT_INCOME_TOTAL <= @upper_bound')

In [74]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   ID                       36457 non-null  int64   
 1   CODE_GENDER              36457 non-null  object  
 2   FLAG_OWN_CAR             36457 non-null  int64   
 3   FLAG_OWN_REALTY          36457 non-null  int64   
 4   CNT_CHILDREN             36457 non-null  int64   
 5   AMT_INCOME_TOTAL         36457 non-null  float64 
 6   NAME_INCOME_TYPE         36457 non-null  object  
 7   NAME_EDUCATION_TYPE      36457 non-null  object  
 8   NAME_FAMILY_STATUS       36457 non-null  object  
 9   NAME_HOUSING_TYPE        36457 non-null  object  
 10  DAYS_BIRTH               36457 non-null  int64   
 11  DAYS_EMPLOYED            36457 non-null  int64   
 12  FLAG_MOBIL               36457 non-null  int64   
 13  FLAG_WORK_PHONE          36457 non-null  int64   
 14  FLAG_P

isi merged dataset yang akan digunakan untuk eda

In [75]:
merged_df.columns.tolist()

['ID',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'FLAG_MOBIL',
 'FLAG_WORK_PHONE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'MONTHS_BALANCE_MIN',
 'MONTHS_BALANCE_MAX',
 'COUNT_LATE_LAST_12M',
 'TARGET',
 'CREDIT_HISTORY_LENGTH',
 'AGE',
 'YEARS_EMPLOYED',
 'INCOME_GROUP',
 'AGE_GROUP',
 'EDUCATION_RANK',
 'AMT_INCOME_TOTAL_CAPPED']

In [76]:
merged_df

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTHS_BALANCE_MIN,MONTHS_BALANCE_MAX,COUNT_LATE_LAST_12M,TARGET,CREDIT_HISTORY_LENGTH,AGE,YEARS_EMPLOYED,INCOME_GROUP,AGE_GROUP,EDUCATION_RANK,AMT_INCOME_TOTAL_CAPPED
0,5008804,M,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,-15,0,0,0,-15,32.9,12.4,Medium,30-39,3,380250.0
1,5008805,M,1,1,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,-14,0,1,0,-14,32.9,12.4,Medium,30-39,3,380250.0
2,5008806,M,1,1,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,-29,0,3,0,-29,58.8,3.1,Low,50-59,1,112500.0
3,5008808,F,0,1,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,-4,0,2,0,-4,52.4,8.4,Medium,50-59,1,270000.0
4,5008809,F,0,1,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,-26,-22,0,0,-4,52.4,8.4,Medium,50-59,1,270000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,M,1,1,0,315000.0,Working,Secondary / secondary special,Married,House / apartment,-17348,-2420,1,0,0,0,Managers,2.0,-11,0,4,1,-11,47.5,6.6,Medium,40-49,1,315000.0
36453,5149834,F,0,1,0,157500.0,Commercial associate,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,-23,0,8,1,-23,33.9,3.6,Low,30-39,3,157500.0
36454,5149838,F,0,1,0,157500.0,Pensioner,Higher education,Married,House / apartment,-12387,-1325,1,0,1,1,Medicine staff,2.0,-32,0,0,1,-32,33.9,3.6,Low,30-39,3,157500.0
36455,5150049,F,0,1,0,283500.0,Working,Secondary / secondary special,Married,House / apartment,-17958,-655,1,0,0,0,Sales staff,2.0,-9,0,10,0,-9,49.2,1.8,Medium,40-49,1,283500.0


save merged dataset ke lokal

In [ ]:
# merged_df.to_csv("D:\File Gaung\Kuliah TIF UB\BCC\Intern 2025\Project\Dataset\processed\merged_dataset.csv", index=False)

<>:1: SyntaxWarning: invalid escape sequence '\F'
<>:1: SyntaxWarning: invalid escape sequence '\F'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13808\2178885684.py:1: SyntaxWarning: invalid escape sequence '\F'
  merged_df.to_csv("D:\File Gaung\Kuliah TIF UB\BCC\Intern 2025\Project\Dataset\processed\merged_dataset.csv", index=False)
